In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

In [21]:
# Download training data from open datasets
training_data = datasets.FashionMNIST(
    root='data',
    train=True,
    download=True,
    transform=ToTensor()
)

# Download test data
test_data = datasets.FashionMNIST(
    root='data',
    train=False,
    download=True,
    transform=ToTensor()
)

In [22]:
batch_size = 64

# Create  data loaders
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C H, W]: {X.shape}")
    print(f"Shape of y: {y.shape}")
    break

Shape of X [N, C H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64])


In [23]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using {device} device')

# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)
print(model)

Using cuda device
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [24]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

In [25]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f'loss: {loss:>7f} [{current:>5d}/{size:>5d}]')

In [26]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f'Test Error: \n Accuracy: {(100 * correct):>0.1f}%, Avg loss: {test_loss:>8f} \n')

In [27]:
epochs = 20
for t in range(epochs):
    print(f'Epoch {t+1}\n--------')
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print('Done')

Epoch 1
--------
loss: 2.307421 [   64/60000]
loss: 2.299232 [ 6464/60000]
loss: 2.276413 [12864/60000]
loss: 2.265737 [19264/60000]
loss: 2.254450 [25664/60000]
loss: 2.228625 [32064/60000]
loss: 2.236444 [38464/60000]
loss: 2.210150 [44864/60000]
loss: 2.211324 [51264/60000]
loss: 2.169665 [57664/60000]
Test Error: 
 Accuracy: 35.9%, Avg loss: 2.175875 

Epoch 2
--------
loss: 2.190568 [   64/60000]
loss: 2.185645 [ 6464/60000]
loss: 2.134523 [12864/60000]
loss: 2.137372 [19264/60000]
loss: 2.097585 [25664/60000]
loss: 2.044099 [32064/60000]
loss: 2.065274 [38464/60000]
loss: 2.003506 [44864/60000]
loss: 2.008929 [51264/60000]
loss: 1.926117 [57664/60000]
Test Error: 
 Accuracy: 55.6%, Avg loss: 1.938984 

Epoch 3
--------
loss: 1.974252 [   64/60000]
loss: 1.950822 [ 6464/60000]
loss: 1.846373 [12864/60000]
loss: 1.865090 [19264/60000]
loss: 1.763190 [25664/60000]
loss: 1.714727 [32064/60000]
loss: 1.721193 [38464/60000]
loss: 1.640046 [44864/60000]
loss: 1.658136 [51264/60000]
loss

In [28]:
torch.save(model.state_dict(), 'model.pth')
print('Saved PyTorch model to model.pth')

Saved PyTorch model to model.pth


In [29]:
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load('model.pth', weights_only=True))

<All keys matched successfully>

In [51]:
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

model.eval()
x, y = test_data[64][0], test_data[64][1]
with torch.no_grad():
    x = x.to(device)
    pred = model(x)
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')

Predicted: "Trouser", Actual: "Trouser"
